In [ ]:
import requests, time, json, os
from datetime import datetime, timedelta, date
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, LongType, IntegerType, DateType
)

# ── API ───────────────────────────────────────────────────────
ALPACA_KEY    = ""
ALPACA_SECRET = ""
BASE_URL      = "https://data.alpaca.markets/v2/stocks/bars"

StatementMeta(, bf6e933f-6614-43a9-a664-8d248a4be7be, 15, Finished, Available, Finished, False)

In [ ]:
HEADERS = {
    "APCA-API-KEY-ID":     ALPACA_KEY,
    "APCA-API-SECRET-KEY": ALPACA_SECRET,
}

In [15]:
# ── WATCHLIST ─────────────────────────────────────────────────
# sector        : top-level GICS sector
# sub_industry  : GICS sub-industry
# tier          : anchor (large-cap liquid) | midtier | highbeta (small/speculative)

WATCHLIST = {

    # ── Pharmaceuticals (top 5 by market cap) ────────────────
    "LLY":  {"name": "Eli Lilly",                 "sector": "Health Care", "sub_industry": "Pharmaceuticals"},
    "JNJ":  {"name": "Johnson & Johnson",         "sector": "Health Care", "sub_industry": "Pharmaceuticals"},
    "ABBV": {"name": "AbbVie",                    "sector": "Health Care", "sub_industry": "Pharmaceuticals"},
    "MRK":  {"name": "Merck & Co",                "sector": "Health Care", "sub_industry": "Pharmaceuticals"},
    "PFE":  {"name": "Pfizer",                    "sector": "Health Care", "sub_industry": "Pharmaceuticals"},

    # ── Biotechnology (top 5 by market cap) ──────────────────
    "AMGN": {"name": "Amgen",                     "sector": "Health Care", "sub_industry": "Biotechnology"},
    "VRTX": {"name": "Vertex Pharmaceuticals",    "sector": "Health Care", "sub_industry": "Biotechnology"},
    "REGN": {"name": "Regeneron",                 "sector": "Health Care", "sub_industry": "Biotechnology"},
    "GILD": {"name": "Gilead Sciences",           "sector": "Health Care", "sub_industry": "Biotechnology"},
    "MRNA": {"name": "Moderna",                   "sector": "Health Care", "sub_industry": "Biotechnology"},

    # ── Health Care Equipment (top 5 by market cap) ──────────
    "ISRG": {"name": "Intuitive Surgical",        "sector": "Health Care", "sub_industry": "Health Care Equipment"},
    "ABT":  {"name": "Abbott Laboratories",       "sector": "Health Care", "sub_industry": "Health Care Equipment"},
    "SYK":  {"name": "Stryker",                   "sector": "Health Care", "sub_industry": "Health Care Equipment"},
    "BSX":  {"name": "Boston Scientific",         "sector": "Health Care", "sub_industry": "Health Care Equipment"},
    "MDT":  {"name": "Medtronic",                 "sector": "Health Care", "sub_industry": "Health Care Equipment"},

    # ── Air Freight & Logistics (top 5 by market cap) ────────
    "UPS":  {"name": "United Parcel Service",     "sector": "Industrials", "sub_industry": "Air Freight & Logistics"},
    "FDX":  {"name": "FedEx",                     "sector": "Industrials", "sub_industry": "Air Freight & Logistics"},
    "EXPD": {"name": "Expeditors International",  "sector": "Industrials", "sub_industry": "Air Freight & Logistics"},
    "CHRW": {"name": "C.H. Robinson",             "sector": "Industrials", "sub_industry": "Air Freight & Logistics"},
    "GXO":  {"name": "GXO Logistics",             "sector": "Industrials", "sub_industry": "Air Freight & Logistics"},

    # ── Marine Ports & Services (top 5 by market cap) ────────
    "ZIM":  {"name": "ZIM Integrated Shipping",   "sector": "Industrials", "sub_industry": "Marine Ports & Services"},
    "MATX": {"name": "Matson Inc",                "sector": "Industrials", "sub_industry": "Marine Ports & Services"},
    "GSL":  {"name": "Global Ship Lease",         "sector": "Industrials", "sub_industry": "Marine Ports & Services"},
    "SFL":  {"name": "SFL Corporation",           "sector": "Industrials", "sub_industry": "Marine Ports & Services"},
    "ESEA": {"name": "Euroseas",                  "sector": "Industrials", "sub_industry": "Marine Ports & Services"},

    # ── Airport Services (top 4 → only 4 exist) ──────────────
    "GATX": {"name": "GATX Corporation",          "sector": "Industrials", "sub_industry": "Airport Services"},
    "AAWW": {"name": "Atlas Air Worldwide",       "sector": "Industrials", "sub_industry": "Airport Services"},
    "SKYW": {"name": "SkyWest Inc",               "sector": "Industrials", "sub_industry": "Airport Services"},
    "FLGT": {"name": "Fulgent Genetics",          "sector": "Industrials", "sub_industry": "Airport Services"},
}
 
TICKERS          = list(WATCHLIST.keys())
PHARMA_TICKERS   = [t for t, v in WATCHLIST.items() if v["sector"] == "Health Care"]
LOGISTICS_TICKERS= [t for t, v in WATCHLIST.items() if v["sector"] == "Industrials"]

print(f"Total tickers      : {len(TICKERS)}")
print(f"Pharmaceuticals    : {len(PHARMA_TICKERS)} — {PHARMA_TICKERS}")
print(f"Air Freight & Log  : {len(LOGISTICS_TICKERS)} — {LOGISTICS_TICKERS}")

StatementMeta(, bf6e933f-6614-43a9-a664-8d248a4be7be, 17, Finished, Available, Finished, False)

Total tickers      : 29
Pharmaceuticals    : 15 — ['LLY', 'JNJ', 'ABBV', 'MRK', 'PFE', 'AMGN', 'VRTX', 'REGN', 'GILD', 'MRNA', 'ISRG', 'ABT', 'SYK', 'BSX', 'MDT']
Air Freight & Log  : 14 — ['UPS', 'FDX', 'EXPD', 'CHRW', 'GXO', 'ZIM', 'MATX', 'GSL', 'SFL', 'ESEA', 'GATX', 'AAWW', 'SKYW', 'FLGT']


In [ ]:
from datetime import datetime, timedelta, timezone
import requests
import time

# ── Config ────────────────────────────────────────────────────────────────────
TODAY = datetime.now(timezone.utc).date()

RANGES = {
    "1min":  {"timeframe": "1Min", "lookback_days": 2,   "incremental_days": 1},
    "5min":  {"timeframe": "5Min", "lookback_days": 30,  "incremental_days": 1},
    "daily": {"timeframe": "1Day", "lookback_days": 180, "incremental_days": 0},
}

BATCH_SIZE   = 100   # multi-symbol endpoint supports up to 100 tickers at once
BATCH_SLEEP  = 1     # pause between batches
BRONZE_TABLE = "bronze_bars_v2"

# ✅ Multi-symbol endpoint — no {ticker} in path
BASE_URL = "https://data.alpaca.markets/v2/stocks/bars"


# ── Fetch ─────────────────────────────────────────────────────────────────────
def fetch_bars_batch(
    tickers: list[str],
    timeframe: str,
    from_date: str,
    to_date: str,
) -> dict[str, list[dict]]:
    """
    Fetch OHLCV bars for multiple tickers in a single API call.
    Handles cursor pagination and retries on 429.
    Returns { ticker: [bar, ...] }
    """
    params = {
        "symbols"   : ",".join(tickers),
        "timeframe" : timeframe,
        "start"     : from_date,
        "end"       : to_date,
        "limit"     : 10000,
        "adjustment": "all",
        "feed"      : "iex",
    }

    all_results: dict[str, list] = {t: [] for t in tickers}

    for attempt in range(3):
        try:
            resp = requests.get(BASE_URL, headers=HEADERS, params=params, timeout=15)

            if resp.status_code == 429:
                wait = 30 * (attempt + 1)
                print(f"  ⚠ 429 rate limit. Waiting {wait}s...")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            data = resp.json()

            # merge first page
            for symbol, bars in (data.get("bars") or {}).items():
                all_results[symbol].extend(bars)

            # paginate
            while data.get("next_page_token"):
                resp = requests.get(
                    BASE_URL,
                    headers=HEADERS,
                    params={**params, "page_token": data["next_page_token"]},
                    timeout=15,
                )
                resp.raise_for_status()
                data = resp.json()
                for symbol, bars in (data.get("bars") or {}).items():
                    all_results[symbol].extend(bars)
                time.sleep(0.1)

            # summary
            found    = {s: len(b) for s, b in all_results.items() if b}
            missing  = [s for s, b in all_results.items() if not b]
            print(f"  ✓ {len(found)} symbols fetched | "
                  f"{'⚠ no data: ' + str(missing) if missing else 'all present'}")
            return all_results

        except requests.RequestException as e:
            print(f"  ✗ Attempt {attempt + 1}/3: {e}")
            time.sleep(5)

    print(f"  ✗ All retries exhausted for batch {tickers[:3]}...")
    return all_results


# ── Orchestrate ───────────────────────────────────────────────────────────────
def fetch_all(tickers: list[str], granularity: str) -> dict[str, list]:
    """
    Fetch one granularity for all tickers using batched multi-symbol calls.
    granularity: '1min' | '5min' | 'daily'
    Returns { ticker: [bars] }
    """
    cfg = RANGES[granularity]

    if INCREMENTAL and cfg["incremental_days"] > 0:
        from_date = (TODAY - timedelta(days=cfg["incremental_days"])).isoformat()
    else:
        from_date = (TODAY - timedelta(days=cfg["lookback_days"])).isoformat()
    to_date = TODAY.isoformat()

    print(f"\n── Fetching {granularity} bars ({from_date} → {to_date}) ──")
    all_results: dict[str, list] = {}

    total_batches = -(-len(tickers) // BATCH_SIZE)

    for i, batch_start in enumerate(range(0, len(tickers), BATCH_SIZE)):
        batch = tickers[batch_start : batch_start + BATCH_SIZE]
        print(f"  Batch {i + 1}/{total_batches}: {len(batch)} symbols")

        batch_results = fetch_bars_batch(batch, cfg["timeframe"], from_date, to_date)
        all_results.update(batch_results)

        if batch_start + BATCH_SIZE < len(tickers):
            print(f"  ⏳ Batch pause: {BATCH_SLEEP}s...")
            time.sleep(BATCH_SLEEP)

    total_bars = sum(len(b) for b in all_results.values())
    print(f"\n  ✅ {granularity} complete — "
          f"{len(all_results)} symbols, {total_bars:,} bars total")
    return all_results

StatementMeta(, bf6e933f-6614-43a9-a664-8d248a4be7be, 18, Finished, Available, Finished, False)

In [17]:
# ── BRONZE SCHEMA ─────────────────────────────────────────────
# Added: sector, sub_industry, tier columns for cross-sector analysis

STRING_SCHEMA = StructType([
    StructField("ticker",       StringType(), True),
    StructField("sector",       StringType(), True),
    StructField("sub_industry", StringType(), True),
    StructField("granularity",  StringType(), True),
    StructField("t",            StringType(), True),
    StructField("o",            StringType(), True),
    StructField("h",            StringType(), True),
    StructField("l",            StringType(), True),
    StructField("c",            StringType(), True),
    StructField("v",            StringType(), True),
    StructField("vw",           StringType(), True),
    StructField("n",            StringType(), True),
])


def write_bronze(raw: dict[str, list], granularity: str):
    """
    Flatten raw Polygon results → Spark DataFrame → append to Delta bronze table.
    Deduplicates on (ticker, granularity, t) before writing.
    Now enriches each row with sector, sub_industry, tier from WATCHLIST.
    """
    rows = [
        {
            "ticker":       ticker,
            "sector":       WATCHLIST[ticker]["sector"],
            "sub_industry": WATCHLIST[ticker]["sub_industry"],
            "granularity":  granularity,
            "t":  str(bar["t"])  if bar.get("t")  is not None else None,
            "o":  str(bar["o"])  if bar.get("o")  is not None else None,
            "h":  str(bar["h"])  if bar.get("h")  is not None else None,
            "l":  str(bar["l"])  if bar.get("l")  is not None else None,
            "c":  str(bar["c"])  if bar.get("c")  is not None else None,
            "v":  str(bar["v"])  if bar.get("v")  is not None else None,
            "vw": str(bar["vw"]) if bar.get("vw") is not None else None,
            "n":  str(bar["n"])  if bar.get("n")  is not None else None,
        }
        for ticker, bars in raw.items()
        for bar in bars
    ]

    if not rows:
        print(f"Bronze [{granularity}]: nothing to write")
        return

    # Step 1: uniform string schema — no type inference
    df_raw = spark.createDataFrame(rows, schema=STRING_SCHEMA)

    # Step 2: cast to target types
    df = (df_raw
        .withColumn("t",  F.col("t") .cast(LongType()))
        .withColumn("o",  F.col("o") .cast(DoubleType()))
        .withColumn("h",  F.col("h") .cast(DoubleType()))
        .withColumn("l",  F.col("l") .cast(DoubleType()))
        .withColumn("c",  F.col("c") .cast(DoubleType()))
        .withColumn("v",  F.col("v") .cast(DoubleType()))
        .withColumn("vw", F.col("vw").cast(DoubleType()))
        .withColumn("n",  F.col("n") .cast(LongType()))
        .select("ticker", "sector", "sub_industry",
                "granularity", "t", "o", "h", "l", "c", "v", "vw", "n")
    )

    # Step 3: deduplicate against existing table
    try:
        existing = spark.read.format("delta").table(BRONZE_TABLE)
        new_rows = df.join(
            existing.filter(F.col("granularity") == granularity)
                    .select("ticker", "granularity", "t"),
            on=["ticker", "granularity", "t"],
            how="left_anti",
        )
    except Exception:
        new_rows = df  # first run — table doesn't exist yet

    n = new_rows.count()
    if n > 0:
        (new_rows.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(BRONZE_TABLE))
        print(f"✓ Bronze [{granularity}]: +{n:,} rows")
    else:
        print(f"✓ Bronze [{granularity}]: already up to date")

StatementMeta(, bf6e933f-6614-43a9-a664-8d248a4be7be, 19, Finished, Available, Finished, False)

In [18]:
# ── PIPELINE MODES ────────────────────────────────────────────
# INCREMENTAL=False  → full backfill (first run)
# INCREMENTAL=True   → only 1min + 5min (set after first run)
# REFRESH_DAILY=True → also refresh daily bars (set once/day after market close)
# SECTOR_FILTER      → None = all tickers | 'Health Care' | 'Industrials'
#                       useful for staggering runs across time windows

INCREMENTAL   = True
SECTOR_FILTER = None   # ← set to 'Health Care' or 'Industrials' to run one sector at a time


def run_pipeline():
    start = datetime.utcnow()

    # Apply sector filter if set
    tickers = (
        [t for t, v in WATCHLIST.items() if v["sector"] == SECTOR_FILTER]
        if SECTOR_FILTER
        else TICKERS
    )

    print("=" * 65)
    print("  Sector Intelligence Pipeline")
    print("  Pharmaceuticals × Air Freight & Logistics")
    print(f"  Mode         : {'Incremental' if INCREMENTAL else 'Full backfill'}")
    print(f"  Sector filter: {SECTOR_FILTER or 'All'}")
    print(f"  Tickers      : {len(tickers)} — {tickers}")
    print(f"  Started      : {start.strftime('%Y-%m-%d %H:%M:%S')} UTC")
    print("=" * 65)

    # ── Step 1: Ingest ────────────────────────────────────────
    print("\n[1/3] Ingesting 1min bars...")
    raw_1min = fetch_all(tickers, "1min")
    write_bronze(raw_1min, "1min")

    print("\n[2/3] Ingesting 5min bars...")
    raw_5min = fetch_all(tickers, "5min")
    write_bronze(raw_5min, "5min")

    if not INCREMENTAL:
        print("\n[3/3] Ingesting daily bars...")
        raw_daily = fetch_all(tickers, "daily")
        write_bronze(raw_daily, "daily")
    else:
        print("\n[3/3] Skipping daily bars (incremental mode, REFRESH_DAILY=False)")

    elapsed = (datetime.utcnow() - start).total_seconds() / 60
    print(f"\n✅ Pipeline complete in {elapsed:.1f} minutes")
    print(f"   Bronze table: {BRONZE_TABLE}")


# ── ENTRY POINT ───────────────────────────────────────────────
run_pipeline()




StatementMeta(, bf6e933f-6614-43a9-a664-8d248a4be7be, 20, Finished, Available, Finished, False)

  Sector Intelligence Pipeline
  Pharmaceuticals × Air Freight & Logistics
  Mode         : Incremental
  Sector filter: All
  Tickers      : 29 — ['LLY', 'JNJ', 'ABBV', 'MRK', 'PFE', 'AMGN', 'VRTX', 'REGN', 'GILD', 'MRNA', 'ISRG', 'ABT', 'SYK', 'BSX', 'MDT', 'UPS', 'FDX', 'EXPD', 'CHRW', 'GXO', 'ZIM', 'MATX', 'GSL', 'SFL', 'ESEA', 'GATX', 'AAWW', 'SKYW', 'FLGT']
  Started      : 2026-06-04 06:45:38 UTC

[1/3] Ingesting 1min bars...

── Fetching 1min bars (2026-06-03 → 2026-06-04) ──
  Batch 1/2: ['LLY', 'JNJ', 'ABBV', 'MRK', 'PFE', 'AMGN', 'VRTX', 'REGN', 'GILD', 'MRNA', 'ISRG', 'ABT', 'SYK', 'BSX', 'MDT', 'UPS', 'FDX', 'EXPD', 'CHRW', 'GXO']
  ✓ LLY    [1Min]: 302 bars
  ✓ JNJ    [1Min]: 305 bars
  ✓ ABBV   [1Min]: 255 bars
  ✓ MRK    [1Min]: 315 bars
  ✓ PFE    [1Min]: 360 bars
  ✓ AMGN   [1Min]: 273 bars
  ✓ VRTX   [1Min]: 238 bars
  ✓ REGN   [1Min]: 284 bars
  ✓ GILD   [1Min]: 354 bars
  ✓ MRNA   [1Min]: 251 bars
  ✓ ISRG   [1Min]: 288 bars
  ✓ ABT    [1Min]: 366 bars
  ✓ SYK    [